# 🚀 LUMEN — Train YOLO11 on RDD2022 Road Damage Dataset (~47k Images)
This notebook downloads the complete **RDD2022 dataset (13.3 GB)**, converts all annotations (Pothole, Alligator Crack, etc.) into YOLO format, and trains **YOLO11 Nano** on a GPU.

> **Note:** Make sure GPU is enabled: `Runtime` ➔ `Change runtime type` ➔ `T4 GPU`.

In [ ]:
# 1. Install dependencies
!pip install -q ultralytics albumentations opencv-python pyyaml tqdm
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2. Download & Extract RDD2022 dataset
import os
import glob
from pathlib import Path

if not os.path.exists("/content/RDD2022.zip"):
    print("⏳ Downloading RDD2022 (13.3 GB) directly to Colab... (~2 mins)")
    !wget -c -O /content/RDD2022.zip "https://ndownloader.figshare.com/files/38030910"
else:
    print("RDD2022.zip already exists.")

if not os.path.exists("/content/rdd2022_raw"):
    print("📦 Extracting RDD2022.zip...")
    !unzip -q /content/RDD2022.zip -d /content/rdd2022_raw/

# Extract country zip files (India, Japan, Czech, United_States, etc.)
country_zips = glob.glob("/content/rdd2022_raw/**/*.zip", recursive=True)
print(f"📦 Extracting {len(country_zips)} country archives...")
for cz in country_zips:
    if os.path.basename(cz) == "RDD2022.zip":
        continue
    cname = Path(cz).stem
    dest_dir = f"/content/rdd2022_raw/extracted/{cname}"
    if not os.path.exists(dest_dir):
        print(f"  → Extracting {cname}...")
        !unzip -q -o "{cz}" -d "{dest_dir}"

print("✅ Extraction Complete!")

In [ ]:
# 3. Index & Convert XML annotations to YOLO format
import os
import glob
import shutil
import random
import xml.etree.ElementTree as ET
from pathlib import Path
from tqdm import tqdm

print("🔎 Indexing images and annotations...")
all_jpgs = glob.glob("/content/rdd2022_raw/**/*.jpg", recursive=True)
img_dict = {Path(p).stem: p for p in all_jpgs}

all_xmls = glob.glob("/content/rdd2022_raw/**/*.xml", recursive=True)
print(f"✅ Found {len(img_dict)} total JPG images and {len(all_xmls)} XML annotations!")

YOLO_DIR = Path("/content/rdd_yolo")
for split in ["train", "val"]:
    os.makedirs(YOLO_DIR / "images" / split, exist_ok=True)
    os.makedirs(YOLO_DIR / "labels" / split, exist_ok=True)

# Class mapping for RDD2022:
CLASS_MAP = {
    "D40": 0,  # Pothole
    "D20": 1,  # Alligator Crack
    "D00": 2,  # Longitudinal Crack
    "D10": 3,  # Transverse Crack
}

def convert_box(size, box):
    dw = 1.0 / size[0]
    dh = 1.0 / size[1]
    x = (box[0] + box[1]) / 2.0 - 1.0
    y = (box[2] + box[3]) / 2.0 - 1.0
    w = box[1] - box[0]
    h = box[3] - box[2]
    return (x * dw, y * dh, w * dw, h * dh)

valid_pairs = []
for xml_p in all_xmls:
    stem = Path(xml_p).stem
    if stem in img_dict:
        valid_pairs.append((xml_p, img_dict[stem]))

print(f"🎯 Matched {len(valid_pairs)} image-annotation pairs!")

random.seed(42)
random.shuffle(valid_pairs)

split_idx = int(len(valid_pairs) * 0.85)
train_pairs = valid_pairs[:split_idx]
val_pairs = valid_pairs[split_idx:]

def process_pairs(pairs, split):
    count = 0
    for xml_path, img_path in tqdm(pairs, desc=f"Processing {split}"):
        try:
            tree = ET.parse(xml_path)
            root = tree.getroot()
            size = root.find("size")
            if size is None:
                continue
            w = int(size.find("width").text)
            h = int(size.find("height").text)
            if w == 0 or h == 0:
                continue
            
            yolo_labels = []
            for obj in root.iter("object"):
                name_elem = obj.find("name")
                if name_elem is not None and name_elem.text in CLASS_MAP:
                    cls_id = CLASS_MAP[name_elem.text]
                    xmlbox = obj.find("bndbox")
                    b = (float(xmlbox.find("xmin").text), float(xmlbox.find("xmax").text),
                         float(xmlbox.find("ymin").text), float(xmlbox.find("ymax").text))
                    bb = convert_box((w, h), b)
                    yolo_labels.append(f"{cls_id} {' '.join([f'{a:.6f}' for a in bb])}")
            
            if yolo_labels:
                img_name = Path(img_path).name
                shutil.copy(img_path, YOLO_DIR / "images" / split / img_name)
                with open(YOLO_DIR / "labels" / split / f"{Path(img_name).stem}.txt", "w") as f:
                    f.write("\n".join(yolo_labels))
                count += 1
        except Exception:
            continue
    print(f"✨ Successfully saved {count} {split} images to YOLO dataset.")

process_pairs(train_pairs, "train")
process_pairs(val_pairs, "val")

# Create data.yaml
yaml_content = """
path: /content/rdd_yolo
train: images/train
val: images/val

names:
  0: Pothole
  1: Alligator Crack
  2: Longitudinal Crack
  3: Transverse Crack
"""
with open("/content/rdd_yolo/data.yaml", "w") as f:
    f.write(yaml_content.strip())

print("\n🚀 Setup Complete! Ready to train.")

In [ ]:
# 4. Train YOLO11 on GPU
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

results = model.train(
    data="/content/rdd_yolo/data.yaml",
    epochs=50,
    imgsz=640,
    batch=32,
    device=0,
    project="rdd_project",
    name="lumen_rdd_model",
    save=True
)

In [ ]:
# 5. Download trained weights (best.pt)
from google.colab import files
import glob

model_weights = glob.glob("/content/rdd_project/**/weights/best.pt", recursive=True)
if model_weights:
    print(f"📥 Downloading {model_weights[0]}...")
    files.download(model_weights[0])
else:
    print("Weights file not found.")